# Gaussian Mixture Models (GMM)

A comprehensive guide to understanding, implementing, and applying Gaussian Mixture Models for clustering and density estimation.

## Table of Contents
1. [Theory Section](#1.-Theory-Section)
2. [Implementation from Scratch](#2.-Implementation-from-Scratch)
3. [Training & Optimization](#3.-Training-&-Optimization)
4. [Diagnostics & Evaluation](#4.-Diagnostics-&-Evaluation)
5. [Visualizations](#5.-Visualizations)
6. [Use Cases & Guidelines](#6.-Use-Cases-&-Guidelines)
7. [Comparison with sklearn](#7.-Comparison-with-sklearn)

---
## 1. Theory Section

### 1.1 Mixture Models

A **mixture model** is a probabilistic model that assumes the data is generated from a mixture of several underlying probability distributions. In Gaussian Mixture Models (GMM), we assume each component is a Gaussian (normal) distribution.

The probability density function of a GMM with K components is:

$$p(x) = \sum_{k=1}^{K} \pi_k \mathcal{N}(x | \mu_k, \Sigma_k)$$

Where:
- $\pi_k$ are the **mixing coefficients** (weights) with $\sum_{k=1}^{K} \pi_k = 1$ and $\pi_k \geq 0$
- $\mu_k$ is the **mean** of component k
- $\Sigma_k$ is the **covariance matrix** of component k
- $\mathcal{N}(x | \mu_k, \Sigma_k)$ is the multivariate Gaussian PDF

### 1.2 The EM Algorithm

The **Expectation-Maximization (EM)** algorithm is used to find maximum likelihood estimates of the GMM parameters. It iteratively alternates between two steps:

#### E-Step (Expectation)
Calculate the **responsibility** (posterior probability) that component k generated data point n:

$$\gamma_{nk} = \frac{\pi_k \mathcal{N}(x_n | \mu_k, \Sigma_k)}{\sum_{j=1}^{K} \pi_j \mathcal{N}(x_n | \mu_j, \Sigma_j)}$$

#### M-Step (Maximization)
Update the parameters using the responsibilities:

$$N_k = \sum_{n=1}^{N} \gamma_{nk}$$ (effective number of points in cluster k)

$$\mu_k^{new} = \frac{1}{N_k} \sum_{n=1}^{N} \gamma_{nk} x_n$$

$$\Sigma_k^{new} = \frac{1}{N_k} \sum_{n=1}^{N} \gamma_{nk} (x_n - \mu_k^{new})(x_n - \mu_k^{new})^T$$

$$\pi_k^{new} = \frac{N_k}{N}$$

### 1.3 Soft Clustering vs Hard Clustering

| Aspect | Hard Clustering (K-Means) | Soft Clustering (GMM) |
|--------|---------------------------|------------------------|
| Assignment | Each point belongs to exactly one cluster | Points have probabilities of belonging to each cluster |
| Output | Cluster labels | Responsibility matrix (probabilities) |
| Cluster Shape | Spherical (Voronoi cells) | Elliptical (covariance-defined) |
| Uncertainty | Not captured | Naturally captured via probabilities |

### 1.4 Covariance Types

Different covariance parameterizations offer trade-offs between flexibility and complexity:

| Type | Description | Parameters per Component | Shape |
|------|-------------|--------------------------|-------|
| **full** | Full covariance matrix | $d(d+1)/2$ | Arbitrary ellipsoid |
| **tied** | Same full covariance for all components | $d(d+1)/2$ total | Same shape, different positions |
| **diag** | Diagonal covariance (axis-aligned) | $d$ | Axis-aligned ellipsoid |
| **spherical** | Single variance for all dimensions | $1$ | Sphere |

### 1.5 Model Selection: BIC and AIC

To select the optimal number of components K, we use information criteria:

**Bayesian Information Criterion (BIC):**
$$BIC = -2 \ln(L) + p \ln(N)$$

**Akaike Information Criterion (AIC):**
$$AIC = -2 \ln(L) + 2p$$

Where:
- $L$ is the maximum likelihood
- $p$ is the number of parameters
- $N$ is the number of samples

**Lower values indicate better models.** BIC penalizes complexity more heavily than AIC.

### 1.6 Time and Space Complexity

For N samples, K components, D dimensions, and I iterations:

| Operation | Time Complexity | Space Complexity |
|-----------|----------------|------------------|
| E-Step | $O(NKD^2)$ | $O(NK)$ for responsibilities |
| M-Step | $O(NKD^2)$ | $O(KD^2)$ for covariances |
| Overall Fit | $O(INKD^2)$ | $O(NK + KD^2)$ |
| Predict | $O(NKD^2)$ | $O(NK)$ |

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scipy import linalg
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

---
## 2. Implementation from Scratch

We implement a complete Gaussian Mixture Model using only NumPy.

In [ ]:
class GaussianMixtureModel:
    """
    Gaussian Mixture Model implementation using the EM algorithm.
    
    Parameters
    ----------
    n_components : int, default=3
        Number of mixture components (clusters).
    covariance_type : str, default='full'
        Type of covariance: 'full', 'tied', 'diag', 'spherical'.
    max_iter : int, default=100
        Maximum number of EM iterations.
    tol : float, default=1e-4
        Convergence threshold for log-likelihood improvement.
    n_init : int, default=1
        Number of initializations; best result is kept.
    init_method : str, default='kmeans'
        Initialization method: 'kmeans' or 'random'.
    reg_covar : float, default=1e-6
        Regularization for covariance matrices.
    random_state : int, default=None
        Random seed for reproducibility.
    
    Attributes
    ----------
    weights_ : array, shape (n_components,)
        Mixing coefficients for each component.
    means_ : array, shape (n_components, n_features)
        Mean of each component.
    covariances_ : array
        Covariance of each component (shape depends on covariance_type).
    converged_ : bool
        True if EM converged before max_iter.
    n_iter_ : int
        Number of iterations performed.
    log_likelihood_history_ : list
        Log-likelihood at each iteration.
    """
    
    def __init__(self, n_components=3, covariance_type='full', max_iter=100,
                 tol=1e-4, n_init=1, init_method='kmeans', reg_covar=1e-6,
                 random_state=None):
        self.n_components = n_components
        self.covariance_type = covariance_type
        self.max_iter = max_iter
        self.tol = tol
        self.n_init = n_init
        self.init_method = init_method
        self.reg_covar = reg_covar
        self.random_state = random_state
        
        # Parameters to be learned
        self.weights_ = None
        self.means_ = None
        self.covariances_ = None
        self.precisions_cholesky_ = None
        
        # Fitting metadata
        self.converged_ = False
        self.n_iter_ = 0
        self.log_likelihood_history_ = []
        self.lower_bound_ = -np.inf
    
    def _multivariate_gaussian_pdf(self, X, mean, precision_chol):
        """
        Compute log probability of X under a multivariate Gaussian.
        Uses Cholesky decomposition for numerical stability.
        
        Parameters
        ----------
        X : array, shape (n_samples, n_features)
        mean : array, shape (n_features,)
        precision_chol : array, shape (n_features, n_features)
            Cholesky decomposition of precision matrix.
        
        Returns
        -------
        log_prob : array, shape (n_samples,)
        """
        n_samples, n_features = X.shape
        
        # Log determinant from Cholesky factor
        log_det = 2 * np.sum(np.log(np.diag(precision_chol)))
        
        # Mahalanobis distance: (x-mu)^T * Precision * (x-mu)
        diff = X - mean
        y = np.dot(diff, precision_chol)
        mahalanobis = np.sum(y ** 2, axis=1)
        
        # Log probability
        log_prob = -0.5 * (n_features * np.log(2 * np.pi) - log_det + mahalanobis)
        
        return log_prob
    
    def _compute_precision_cholesky(self, covariances):
        """
        Compute Cholesky decomposition of precision matrices.
        
        Parameters
        ----------
        covariances : array
            Covariance matrices.
        
        Returns
        -------
        precisions_chol : array
            Cholesky decomposition of precision matrices.
        """
        n_components = self.n_components
        n_features = covariances.shape[-1] if covariances.ndim > 1 else 1
        
        if self.covariance_type == 'full':
            precisions_chol = np.empty((n_components, n_features, n_features))
            for k in range(n_components):
                try:
                    cov_chol = linalg.cholesky(covariances[k], lower=True)
                    precisions_chol[k] = linalg.solve_triangular(
                        cov_chol, np.eye(n_features), lower=True
                    ).T
                except linalg.LinAlgError:
                    raise ValueError(f"Covariance matrix {k} is not positive definite")
        
        elif self.covariance_type == 'tied':
            try:
                cov_chol = linalg.cholesky(covariances, lower=True)
                precisions_chol = linalg.solve_triangular(
                    cov_chol, np.eye(n_features), lower=True
                ).T
            except linalg.LinAlgError:
                raise ValueError("Tied covariance matrix is not positive definite")
        
        elif self.covariance_type == 'diag':
            precisions_chol = 1.0 / np.sqrt(covariances)
        
        elif self.covariance_type == 'spherical':
            precisions_chol = 1.0 / np.sqrt(covariances)
        
        return precisions_chol
    
    def _estimate_log_prob(self, X):
        """
        Compute weighted log probabilities for each sample and component.
        
        Parameters
        ----------
        X : array, shape (n_samples, n_features)
        
        Returns
        -------
        weighted_log_prob : array, shape (n_samples, n_components)
        """
        n_samples, n_features = X.shape
        n_components = self.n_components
        
        log_prob = np.empty((n_samples, n_components))
        
        for k in range(n_components):
            if self.covariance_type == 'full':
                precision_chol = self.precisions_cholesky_[k]
            elif self.covariance_type == 'tied':
                precision_chol = self.precisions_cholesky_
            elif self.covariance_type == 'diag':
                precision_chol = np.diag(self.precisions_cholesky_[k])
            elif self.covariance_type == 'spherical':
                precision_chol = np.eye(n_features) * self.precisions_cholesky_[k]
            
            log_prob[:, k] = self._multivariate_gaussian_pdf(
                X, self.means_[k], precision_chol
            )
        
        # Add log weights
        weighted_log_prob = log_prob + np.log(self.weights_)
        
        return weighted_log_prob
    
    def _e_step(self, X):
        """
        E-step: Compute responsibilities (posterior probabilities).
        
        Parameters
        ----------
        X : array, shape (n_samples, n_features)
        
        Returns
        -------
        log_likelihood : float
            Mean log-likelihood of samples.
        responsibilities : array, shape (n_samples, n_components)
            Posterior probabilities.
        """
        weighted_log_prob = self._estimate_log_prob(X)
        
        # Log-sum-exp for numerical stability
        log_prob_norm = np.logaddexp.reduce(weighted_log_prob, axis=1)
        
        # Mean log-likelihood
        log_likelihood = np.mean(log_prob_norm)
        
        # Responsibilities (normalized)
        with np.errstate(under='ignore'):
            responsibilities = np.exp(weighted_log_prob - log_prob_norm[:, np.newaxis])
        
        return log_likelihood, responsibilities
    
    def _m_step(self, X, responsibilities):
        """
        M-step: Update parameters based on responsibilities.
        
        Parameters
        ----------
        X : array, shape (n_samples, n_features)
        responsibilities : array, shape (n_samples, n_components)
        """
        n_samples, n_features = X.shape
        
        # Effective number of points per component
        nk = responsibilities.sum(axis=0) + 1e-10  # Avoid division by zero
        
        # Update weights
        self.weights_ = nk / n_samples
        
        # Update means
        self.means_ = np.dot(responsibilities.T, X) / nk[:, np.newaxis]
        
        # Update covariances
        self._estimate_covariances(X, responsibilities, nk)
        
        # Compute precision Cholesky
        self.precisions_cholesky_ = self._compute_precision_cholesky(self.covariances_)
    
    def _estimate_covariances(self, X, responsibilities, nk):
        """
        Estimate covariance matrices based on covariance type.
        
        Parameters
        ----------
        X : array, shape (n_samples, n_features)
        responsibilities : array, shape (n_samples, n_components)
        nk : array, shape (n_components,)
            Effective number of points per component.
        """
        n_samples, n_features = X.shape
        n_components = self.n_components
        
        if self.covariance_type == 'full':
            self.covariances_ = np.empty((n_components, n_features, n_features))
            for k in range(n_components):
                diff = X - self.means_[k]
                self.covariances_[k] = (
                    np.dot(responsibilities[:, k] * diff.T, diff) / nk[k]
                    + self.reg_covar * np.eye(n_features)
                )
        
        elif self.covariance_type == 'tied':
            self.covariances_ = np.zeros((n_features, n_features))
            for k in range(n_components):
                diff = X - self.means_[k]
                self.covariances_ += np.dot(responsibilities[:, k] * diff.T, diff)
            self.covariances_ /= n_samples
            self.covariances_ += self.reg_covar * np.eye(n_features)
        
        elif self.covariance_type == 'diag':
            self.covariances_ = np.empty((n_components, n_features))
            for k in range(n_components):
                diff = X - self.means_[k]
                self.covariances_[k] = (
                    np.dot(responsibilities[:, k], diff ** 2) / nk[k]
                    + self.reg_covar
                )
        
        elif self.covariance_type == 'spherical':
            self.covariances_ = np.empty(n_components)
            for k in range(n_components):
                diff = X - self.means_[k]
                self.covariances_[k] = (
                    np.dot(responsibilities[:, k], np.sum(diff ** 2, axis=1)) /
                    (nk[k] * n_features)
                    + self.reg_covar
                )
    
    def _initialize_parameters(self, X):
        """
        Initialize GMM parameters.
        
        Parameters
        ----------
        X : array, shape (n_samples, n_features)
        """
        n_samples, n_features = X.shape
        
        if self.random_state is not None:
            np.random.seed(self.random_state)
        
        # Initialize weights uniformly
        self.weights_ = np.ones(self.n_components) / self.n_components
        
        # Initialize means
        if self.init_method == 'kmeans':
            kmeans = KMeans(n_clusters=self.n_components, n_init=1,
                          random_state=self.random_state)
            kmeans.fit(X)
            self.means_ = kmeans.cluster_centers_
            resp = np.zeros((n_samples, self.n_components))
            resp[np.arange(n_samples), kmeans.labels_] = 1
        else:  # random
            indices = np.random.choice(n_samples, self.n_components, replace=False)
            self.means_ = X[indices].copy()
            resp = np.random.rand(n_samples, self.n_components)
            resp /= resp.sum(axis=1, keepdims=True)
        
        # Initialize covariances using initial responsibilities
        nk = resp.sum(axis=0) + 1e-10
        self._estimate_covariances(X, resp, nk)
        self.precisions_cholesky_ = self._compute_precision_cholesky(self.covariances_)
    
    def fit(self, X):
        """
        Fit the GMM to data using EM algorithm.
        
        Parameters
        ----------
        X : array, shape (n_samples, n_features)
            Training data.
        
        Returns
        -------
        self
        """
        X = np.array(X, dtype=np.float64)
        n_samples, n_features = X.shape
        
        best_lower_bound = -np.inf
        best_params = None
        
        for init in range(self.n_init):
            # Set seed for this initialization
            if self.random_state is not None:
                np.random.seed(self.random_state + init)
            
            # Initialize parameters
            self._initialize_parameters(X)
            
            # EM iterations
            log_likelihood_history = []
            lower_bound = -np.inf
            
            for iteration in range(self.max_iter):
                # E-step
                log_likelihood, responsibilities = self._e_step(X)
                log_likelihood_history.append(log_likelihood)
                
                # Check convergence
                change = log_likelihood - lower_bound
                if abs(change) < self.tol:
                    converged = True
                    break
                lower_bound = log_likelihood
                
                # M-step
                self._m_step(X, responsibilities)
            else:
                converged = False
            
            # Keep best result
            if lower_bound > best_lower_bound:
                best_lower_bound = lower_bound
                best_params = {
                    'weights': self.weights_.copy(),
                    'means': self.means_.copy(),
                    'covariances': self.covariances_.copy(),
                    'precisions_cholesky': self.precisions_cholesky_.copy(),
                    'converged': converged,
                    'n_iter': iteration + 1,
                    'log_likelihood_history': log_likelihood_history
                }
        
        # Set best parameters
        self.weights_ = best_params['weights']
        self.means_ = best_params['means']
        self.covariances_ = best_params['covariances']
        self.precisions_cholesky_ = best_params['precisions_cholesky']
        self.converged_ = best_params['converged']
        self.n_iter_ = best_params['n_iter']
        self.log_likelihood_history_ = best_params['log_likelihood_history']
        self.lower_bound_ = best_lower_bound
        
        return self
    
    def predict(self, X):
        """
        Predict cluster labels for samples (hard assignment).
        
        Parameters
        ----------
        X : array, shape (n_samples, n_features)
        
        Returns
        -------
        labels : array, shape (n_samples,)
            Component labels.
        """
        return self.predict_proba(X).argmax(axis=1)
    
    def predict_proba(self, X):
        """
        Predict posterior probabilities (soft assignment).
        
        Parameters
        ----------
        X : array, shape (n_samples, n_features)
        
        Returns
        -------
        responsibilities : array, shape (n_samples, n_components)
            Posterior probabilities for each component.
        """
        X = np.array(X, dtype=np.float64)
        _, responsibilities = self._e_step(X)
        return responsibilities
    
    def score(self, X):
        """
        Compute mean log-likelihood of samples.
        
        Parameters
        ----------
        X : array, shape (n_samples, n_features)
        
        Returns
        -------
        log_likelihood : float
        """
        X = np.array(X, dtype=np.float64)
        log_likelihood, _ = self._e_step(X)
        return log_likelihood
    
    def score_samples(self, X):
        """
        Compute log-likelihood of each sample.
        
        Parameters
        ----------
        X : array, shape (n_samples, n_features)
        
        Returns
        -------
        log_likelihood : array, shape (n_samples,)
        """
        X = np.array(X, dtype=np.float64)
        weighted_log_prob = self._estimate_log_prob(X)
        return np.logaddexp.reduce(weighted_log_prob, axis=1)
    
    def _n_parameters(self):
        """
        Compute number of free parameters.
        
        Returns
        -------
        n_params : int
        """
        n_features = self.means_.shape[1]
        n_components = self.n_components
        
        # Means: n_components * n_features
        # Weights: n_components - 1 (sum to 1 constraint)
        n_params = n_components * n_features + (n_components - 1)
        
        # Covariances depend on type
        if self.covariance_type == 'full':
            n_params += n_components * n_features * (n_features + 1) // 2
        elif self.covariance_type == 'tied':
            n_params += n_features * (n_features + 1) // 2
        elif self.covariance_type == 'diag':
            n_params += n_components * n_features
        elif self.covariance_type == 'spherical':
            n_params += n_components
        
        return n_params
    
    def bic(self, X):
        """
        Bayesian Information Criterion.
        
        Parameters
        ----------
        X : array, shape (n_samples, n_features)
        
        Returns
        -------
        bic : float
            Lower is better.
        """
        n_samples = X.shape[0]
        return (-2 * self.score(X) * n_samples +
                self._n_parameters() * np.log(n_samples))
    
    def aic(self, X):
        """
        Akaike Information Criterion.
        
        Parameters
        ----------
        X : array, shape (n_samples, n_features)
        
        Returns
        -------
        aic : float
            Lower is better.
        """
        n_samples = X.shape[0]
        return -2 * self.score(X) * n_samples + 2 * self._n_parameters()


print("GaussianMixtureModel class implemented successfully!")

---
## 3. Training & Optimization

### 3.1 Dataset Generation

In [ ]:
def generate_elongated_clusters(n_samples=300, random_state=42):
    """
    Generate data with elongated (elliptical) clusters.
    This demonstrates GMM's advantage over K-Means.
    
    Parameters
    ----------
    n_samples : int
        Total number of samples.
    random_state : int
        Random seed.
    
    Returns
    -------
    X : array, shape (n_samples, 2)
    y : array, shape (n_samples,)
    """
    np.random.seed(random_state)
    n_per_cluster = n_samples // 3
    
    # Cluster 1: Elongated along diagonal (positive correlation)
    cov1 = [[2.0, 1.5], [1.5, 2.0]]
    X1 = np.random.multivariate_normal([0, 0], cov1, n_per_cluster)
    
    # Cluster 2: Elongated along anti-diagonal (negative correlation)
    cov2 = [[2.0, -1.5], [-1.5, 2.0]]
    X2 = np.random.multivariate_normal([5, 5], cov2, n_per_cluster)
    
    # Cluster 3: Elongated horizontally
    cov3 = [[3.0, 0.0], [0.0, 0.3]]
    X3 = np.random.multivariate_normal([2.5, -3], cov3, n_per_cluster)
    
    X = np.vstack([X1, X2, X3])
    y = np.array([0] * n_per_cluster + [1] * n_per_cluster + [2] * n_per_cluster)
    
    return X, y


# Generate datasets
# Standard blobs (spherical clusters)
X_blobs, y_blobs = make_blobs(n_samples=300, centers=3, cluster_std=1.0,
                               random_state=42)

# Elongated clusters
X_elongated, y_elongated = generate_elongated_clusters(n_samples=300, random_state=42)

# Visualize datasets
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_blobs, cmap='viridis',
                alpha=0.7, edgecolors='white', s=50)
axes[0].set_title('Standard Blobs (Spherical Clusters)', fontsize=14)
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

axes[1].scatter(X_elongated[:, 0], X_elongated[:, 1], c=y_elongated,
                cmap='viridis', alpha=0.7, edgecolors='white', s=50)
axes[1].set_title('Elongated Clusters (Elliptical)', fontsize=14)
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

### 3.2 Training the GMM

In [ ]:
# Train GMM on standard blobs
gmm_blobs = GaussianMixtureModel(
    n_components=3,
    covariance_type='full',
    max_iter=100,
    tol=1e-4,
    n_init=3,
    random_state=42
)
gmm_blobs.fit(X_blobs)

print("=== GMM on Standard Blobs ===")
print(f"Converged: {gmm_blobs.converged_}")
print(f"Iterations: {gmm_blobs.n_iter_}")
print(f"Final log-likelihood: {gmm_blobs.lower_bound_:.4f}")
print(f"\nLearned weights: {gmm_blobs.weights_}")
print(f"\nLearned means:\n{gmm_blobs.means_}")

In [ ]:
# Train GMM on elongated clusters
gmm_elongated = GaussianMixtureModel(
    n_components=3,
    covariance_type='full',
    max_iter=100,
    tol=1e-4,
    n_init=3,
    random_state=42
)
gmm_elongated.fit(X_elongated)

print("=== GMM on Elongated Clusters ===")
print(f"Converged: {gmm_elongated.converged_}")
print(f"Iterations: {gmm_elongated.n_iter_}")
print(f"Final log-likelihood: {gmm_elongated.lower_bound_:.4f}")
print(f"\nLearned weights: {gmm_elongated.weights_}")

### 3.3 Effect of Different Covariance Types

In [ ]:
# Compare covariance types on elongated data
covariance_types = ['full', 'tied', 'diag', 'spherical']
results = {}

for cov_type in covariance_types:
    gmm = GaussianMixtureModel(
        n_components=3,
        covariance_type=cov_type,
        max_iter=100,
        n_init=3,
        random_state=42
    )
    gmm.fit(X_elongated)
    results[cov_type] = {
        'model': gmm,
        'bic': gmm.bic(X_elongated),
        'aic': gmm.aic(X_elongated),
        'log_likelihood': gmm.lower_bound_
    }
    print(f"{cov_type:10s} - BIC: {results[cov_type]['bic']:.2f}, "
          f"AIC: {results[cov_type]['aic']:.2f}, "
          f"LogL: {results[cov_type]['log_likelihood']:.4f}")

---
## 4. Diagnostics & Evaluation

### 4.1 Log-Likelihood Convergence

In [ ]:
# Plot convergence for both datasets
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Blobs convergence
axes[0].plot(gmm_blobs.log_likelihood_history_, 'b-o', markersize=4)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Log-Likelihood')
axes[0].set_title('EM Convergence - Standard Blobs')
axes[0].axhline(y=gmm_blobs.lower_bound_, color='r', linestyle='--',
                label=f'Final: {gmm_blobs.lower_bound_:.4f}')
axes[0].legend()

# Elongated convergence
axes[1].plot(gmm_elongated.log_likelihood_history_, 'g-o', markersize=4)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Log-Likelihood')
axes[1].set_title('EM Convergence - Elongated Clusters')
axes[1].axhline(y=gmm_elongated.lower_bound_, color='r', linestyle='--',
                label=f'Final: {gmm_elongated.lower_bound_:.4f}')
axes[1].legend()

plt.tight_layout()
plt.show()

### 4.2 BIC/AIC for Component Selection

In [ ]:
def select_n_components(X, max_components=10, covariance_type='full',
                        random_state=42):
    """
    Select optimal number of components using BIC and AIC.
    
    Parameters
    ----------
    X : array, shape (n_samples, n_features)
    max_components : int
        Maximum number of components to try.
    covariance_type : str
        Type of covariance.
    random_state : int
        Random seed.
    
    Returns
    -------
    results : dict
        BIC and AIC scores for each number of components.
    """
    n_components_range = range(1, max_components + 1)
    bic_scores = []
    aic_scores = []
    
    for n in n_components_range:
        gmm = GaussianMixtureModel(
            n_components=n,
            covariance_type=covariance_type,
            max_iter=100,
            n_init=3,
            random_state=random_state
        )
        gmm.fit(X)
        bic_scores.append(gmm.bic(X))
        aic_scores.append(gmm.aic(X))
    
    return {
        'n_components': list(n_components_range),
        'bic': bic_scores,
        'aic': aic_scores
    }


# Component selection on elongated data
selection_results = select_n_components(X_elongated, max_components=8)

# Plot results
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(selection_results['n_components'], selection_results['bic'],
        'b-o', label='BIC', linewidth=2, markersize=8)
ax.plot(selection_results['n_components'], selection_results['aic'],
        'r-s', label='AIC', linewidth=2, markersize=8)

# Mark optimal
best_bic_idx = np.argmin(selection_results['bic'])
best_aic_idx = np.argmin(selection_results['aic'])

ax.axvline(x=selection_results['n_components'][best_bic_idx],
           color='b', linestyle='--', alpha=0.5,
           label=f'Best BIC: {selection_results["n_components"][best_bic_idx]} components')
ax.axvline(x=selection_results['n_components'][best_aic_idx],
           color='r', linestyle='--', alpha=0.5,
           label=f'Best AIC: {selection_results["n_components"][best_aic_idx]} components')

ax.set_xlabel('Number of Components', fontsize=12)
ax.set_ylabel('Information Criterion Score', fontsize=12)
ax.set_title('Model Selection: BIC vs AIC', fontsize=14)
ax.legend(loc='best')
ax.set_xticks(selection_results['n_components'])

plt.tight_layout()
plt.show()

print(f"\nOptimal number of components:")
print(f"  According to BIC: {selection_results['n_components'][best_bic_idx]}")
print(f"  According to AIC: {selection_results['n_components'][best_aic_idx]}")

---
## 5. Visualizations

### 5.1 Cluster Assignments with Probability Ellipses

In [ ]:
def draw_ellipse(mean, cov, ax, n_std=2.0, **kwargs):
    """
    Draw an ellipse representing a Gaussian component.
    
    Parameters
    ----------
    mean : array, shape (2,)
        Center of ellipse.
    cov : array, shape (2, 2)
        Covariance matrix.
    ax : matplotlib axis
    n_std : float
        Number of standard deviations for ellipse radius.
    **kwargs : additional arguments for Ellipse patch.
    """
    # Eigenvalue decomposition
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    
    # Sort by eigenvalue (largest first)
    order = eigenvalues.argsort()[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]
    
    # Angle of rotation
    angle = np.degrees(np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0]))
    
    # Width and height (2 * n_std * sqrt(eigenvalue))
    width, height = 2 * n_std * np.sqrt(eigenvalues)
    
    ellipse = Ellipse(xy=mean, width=width, height=height, angle=angle, **kwargs)
    ax.add_patch(ellipse)


def plot_gmm_results(X, gmm, ax, title='GMM Clustering'):
    """
    Plot GMM clustering results with probability ellipses.
    
    Parameters
    ----------
    X : array, shape (n_samples, n_features)
    gmm : GaussianMixtureModel
        Fitted GMM.
    ax : matplotlib axis
    title : str
    """
    # Predict labels and probabilities
    labels = gmm.predict(X)
    proba = gmm.predict_proba(X)
    
    # Color map
    colors = plt.cm.viridis(np.linspace(0, 1, gmm.n_components))
    
    # Plot points with colors based on predicted cluster
    for k in range(gmm.n_components):
        mask = labels == k
        ax.scatter(X[mask, 0], X[mask, 1], c=[colors[k]], alpha=0.6,
                  edgecolors='white', s=50, label=f'Cluster {k}')
    
    # Draw ellipses for each component
    for k in range(gmm.n_components):
        if gmm.covariance_type == 'full':
            cov = gmm.covariances_[k]
        elif gmm.covariance_type == 'tied':
            cov = gmm.covariances_
        elif gmm.covariance_type == 'diag':
            cov = np.diag(gmm.covariances_[k])
        elif gmm.covariance_type == 'spherical':
            cov = np.eye(2) * gmm.covariances_[k]
        
        # Draw 1, 2, and 3 standard deviation ellipses
        for n_std, alpha in [(1, 0.3), (2, 0.2), (3, 0.1)]:
            draw_ellipse(gmm.means_[k], cov, ax, n_std=n_std,
                        facecolor=colors[k], alpha=alpha, edgecolor='black',
                        linewidth=1)
        
        # Mark center
        ax.scatter(gmm.means_[k, 0], gmm.means_[k, 1], c='red',
                  marker='x', s=200, linewidths=3, zorder=10)
    
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.legend(loc='best')


# Plot GMM results on both datasets
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

plot_gmm_results(X_blobs, gmm_blobs, axes[0], 'GMM on Standard Blobs')
plot_gmm_results(X_elongated, gmm_elongated, axes[1], 'GMM on Elongated Clusters')

plt.tight_layout()
plt.show()

### 5.2 Comparison with K-Means on Elongated Clusters

In [ ]:
# Fit K-Means on elongated data
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_elongated)

# Compare GMM vs K-Means
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Ground truth
axes[0].scatter(X_elongated[:, 0], X_elongated[:, 1], c=y_elongated,
                cmap='viridis', alpha=0.7, edgecolors='white', s=50)
axes[0].set_title('Ground Truth', fontsize=14)
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

# K-Means
axes[1].scatter(X_elongated[:, 0], X_elongated[:, 1], c=kmeans_labels,
                cmap='viridis', alpha=0.7, edgecolors='white', s=50)
axes[1].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
                c='red', marker='X', s=300, edgecolors='black', linewidths=2)
axes[1].set_title('K-Means Clustering', fontsize=14)
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')

# GMM
plot_gmm_results(X_elongated, gmm_elongated, axes[2], 'GMM Clustering')

plt.tight_layout()
plt.show()

# Calculate accuracy (accounting for label permutation)
from scipy.optimize import linear_sum_assignment

def cluster_accuracy(y_true, y_pred):
    """Calculate clustering accuracy using Hungarian algorithm."""
    confusion = np.zeros((3, 3))
    for t, p in zip(y_true, y_pred):
        confusion[t, p] += 1
    row_ind, col_ind = linear_sum_assignment(-confusion)
    return confusion[row_ind, col_ind].sum() / len(y_true)

gmm_labels = gmm_elongated.predict(X_elongated)
print(f"\nClustering Accuracy on Elongated Data:")
print(f"  K-Means: {cluster_accuracy(y_elongated, kmeans_labels):.4f}")
print(f"  GMM:     {cluster_accuracy(y_elongated, gmm_labels):.4f}")

### 5.3 Soft vs Hard Clustering Visualization

In [ ]:
def plot_soft_clustering(X, gmm, ax):
    """
    Visualize soft clustering with color mixing based on probabilities.
    
    Parameters
    ----------
    X : array, shape (n_samples, n_features)
    gmm : GaussianMixtureModel
    ax : matplotlib axis
    """
    proba = gmm.predict_proba(X)
    
    # Base colors for each cluster (RGB)
    base_colors = np.array([
        [1, 0, 0],  # Red
        [0, 1, 0],  # Green
        [0, 0, 1],  # Blue
    ])
    
    # Mix colors based on probabilities
    mixed_colors = np.dot(proba, base_colors)
    
    ax.scatter(X[:, 0], X[:, 1], c=mixed_colors, alpha=0.7,
              edgecolors='white', s=50)
    ax.set_title('Soft Clustering (Color = Probability Mix)', fontsize=14)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')


fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Hard clustering
labels = gmm_elongated.predict(X_elongated)
axes[0].scatter(X_elongated[:, 0], X_elongated[:, 1], c=labels,
                cmap='Set1', alpha=0.7, edgecolors='white', s=50)
axes[0].set_title('Hard Clustering (argmax)', fontsize=14)
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

# Soft clustering
plot_soft_clustering(X_elongated, gmm_elongated, axes[1])

plt.tight_layout()
plt.show()

### 5.4 Decision Boundaries

In [ ]:
def plot_decision_boundary(X, model, ax, resolution=200, title='Decision Boundary'):
    """
    Plot decision boundaries for clustering.
    
    Parameters
    ----------
    X : array, shape (n_samples, n_features)
    model : fitted model with predict method
    ax : matplotlib axis
    resolution : int
        Grid resolution.
    title : str
    """
    # Create mesh grid
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, resolution),
        np.linspace(y_min, y_max, resolution)
    )
    
    # Predict on grid
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid_points).reshape(xx.shape)
    
    # Plot decision regions
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
    ax.contour(xx, yy, Z, colors='black', linewidths=0.5)
    
    # Plot data points
    labels = model.predict(X)
    ax.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis',
              edgecolors='white', s=50)
    
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)


# Compare decision boundaries: K-Means vs GMM
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Create a simple wrapper for KMeans to match GMM interface
class KMeansWrapper:
    def __init__(self, kmeans):
        self.kmeans = kmeans
    def predict(self, X):
        return self.kmeans.predict(X)

plot_decision_boundary(X_elongated, KMeansWrapper(kmeans), axes[0],
                      title='K-Means Decision Boundaries')
plot_decision_boundary(X_elongated, gmm_elongated, axes[1],
                      title='GMM Decision Boundaries')

plt.tight_layout()
plt.show()

---
## 6. Use Cases & Guidelines

### 6.1 When to Use GMM

**Ideal Use Cases:**

1. **Soft Clustering Required**
   - When you need probability estimates of cluster membership
   - Applications: customer segmentation (customers may belong to multiple segments)

2. **Elliptical/Non-Spherical Clusters**
   - When clusters have different shapes and orientations
   - K-Means assumes spherical clusters; GMM handles elliptical ones

3. **Density Estimation**
   - GMM provides a full probability density model
   - Can be used for anomaly detection (low probability = anomaly)

4. **Generative Modeling**
   - Can sample new data points from the learned distribution
   - Useful for data augmentation

5. **Initialization for Other Algorithms**
   - GMM can provide initial estimates for more complex models

### 6.2 When NOT to Use GMM

**Avoid GMM When:**

1. **Many Components Required**
   - Computational cost grows with K (number of components)
   - Full covariance: O(K * D^2) parameters per component

2. **Non-Gaussian Clusters**
   - GMM assumes Gaussian distributions
   - Will not capture multi-modal or highly skewed clusters

3. **High Dimensions**
   - Covariance estimation becomes unreliable in high-D
   - Consider using diagonal covariance or dimensionality reduction

4. **Very Large Datasets**
   - EM can be slow; consider mini-batch variants or K-Means

5. **Clusters with Very Different Sizes**
   - Small clusters may be absorbed by larger ones
   - May need careful initialization

### 6.3 Component Selection Strategies

In [ ]:
# Demonstration of different strategies
print("="*60)
print("COMPONENT SELECTION STRATEGIES")
print("="*60)

print("""
1. BIC (Bayesian Information Criterion)
   - More conservative; penalizes complexity heavily
   - Preferred when you want simpler models
   - Formula: BIC = -2*ln(L) + p*ln(N)

2. AIC (Akaike Information Criterion)
   - Less conservative; allows more complex models
   - Better for predictive accuracy
   - Formula: AIC = -2*ln(L) + 2p

3. Elbow Method (Log-Likelihood)
   - Plot log-likelihood vs K
   - Look for "elbow" where improvement slows

4. Cross-Validation
   - Split data into train/validation
   - Choose K that maximizes validation log-likelihood

5. Domain Knowledge
   - If you know the true number of clusters, use it
   - Consider business requirements
""")

# Demonstrate cross-validation approach
from sklearn.model_selection import KFold

def cross_validate_gmm(X, n_components_range, n_folds=5, random_state=42):
    """
    Cross-validate GMM for component selection.
    
    Parameters
    ----------
    X : array, shape (n_samples, n_features)
    n_components_range : range
        Range of component numbers to try.
    n_folds : int
        Number of cross-validation folds.
    random_state : int
        Random seed.
    
    Returns
    -------
    results : dict
        Mean and std of log-likelihood for each K.
    """
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    
    results = {'n_components': [], 'mean_score': [], 'std_score': []}
    
    for n in n_components_range:
        scores = []
        for train_idx, val_idx in kf.split(X):
            X_train, X_val = X[train_idx], X[val_idx]
            
            gmm = GaussianMixtureModel(
                n_components=n,
                covariance_type='full',
                max_iter=100,
                n_init=1,
                random_state=random_state
            )
            gmm.fit(X_train)
            scores.append(gmm.score(X_val))
        
        results['n_components'].append(n)
        results['mean_score'].append(np.mean(scores))
        results['std_score'].append(np.std(scores))
    
    return results


# Run cross-validation
cv_results = cross_validate_gmm(X_elongated, range(1, 8))

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

ax.errorbar(cv_results['n_components'], cv_results['mean_score'],
            yerr=cv_results['std_score'], fmt='o-', capsize=5,
            linewidth=2, markersize=8)

best_idx = np.argmax(cv_results['mean_score'])
ax.axvline(x=cv_results['n_components'][best_idx], color='r',
           linestyle='--', label=f'Best: {cv_results["n_components"][best_idx]} components')

ax.set_xlabel('Number of Components', fontsize=12)
ax.set_ylabel('Cross-Validated Log-Likelihood', fontsize=12)
ax.set_title('Cross-Validation for Component Selection', fontsize=14)
ax.legend()
ax.set_xticks(cv_results['n_components'])

plt.tight_layout()
plt.show()

---
## 7. Comparison with sklearn

### 7.1 API Comparison

In [ ]:
from sklearn.mixture import GaussianMixture

# Fit sklearn GMM
sklearn_gmm = GaussianMixture(
    n_components=3,
    covariance_type='full',
    max_iter=100,
    tol=1e-4,
    n_init=3,
    random_state=42
)
sklearn_gmm.fit(X_elongated)

# Fit our GMM
our_gmm = GaussianMixtureModel(
    n_components=3,
    covariance_type='full',
    max_iter=100,
    tol=1e-4,
    n_init=3,
    random_state=42
)
our_gmm.fit(X_elongated)

print("="*60)
print("COMPARISON: Our Implementation vs sklearn")
print("="*60)

print(f"\n{'Metric':<30} {'Ours':>15} {'sklearn':>15}")
print("-"*60)
print(f"{'Converged':<30} {str(our_gmm.converged_):>15} {str(sklearn_gmm.converged_):>15}")
print(f"{'Iterations':<30} {our_gmm.n_iter_:>15} {sklearn_gmm.n_iter_:>15}")
print(f"{'Log-Likelihood':<30} {our_gmm.lower_bound_:>15.4f} {sklearn_gmm.lower_bound_:>15.4f}")
print(f"{'BIC':<30} {our_gmm.bic(X_elongated):>15.2f} {sklearn_gmm.bic(X_elongated):>15.2f}")
print(f"{'AIC':<30} {our_gmm.aic(X_elongated):>15.2f} {sklearn_gmm.aic(X_elongated):>15.2f}")

### 7.2 Results Comparison

In [ ]:
# Compare predictions
our_labels = our_gmm.predict(X_elongated)
sklearn_labels = sklearn_gmm.predict(X_elongated)

# Due to possible label permutation, use accuracy with Hungarian algorithm
agreement = cluster_accuracy(our_labels, sklearn_labels)
print(f"\nLabel agreement between implementations: {agreement:.2%}")

# Compare probability outputs
our_proba = our_gmm.predict_proba(X_elongated)
sklearn_proba = sklearn_gmm.predict_proba(X_elongated)

# Note: Labels may be permuted, so we match by finding best permutation
print(f"\nSample probability comparison (first 5 samples):")
print(f"{'Sample':<10} {'Our Proba (sorted)':>35} {'sklearn Proba (sorted)':>35}")
print("-"*80)
for i in range(5):
    our_sorted = np.sort(our_proba[i])[::-1]
    sk_sorted = np.sort(sklearn_proba[i])[::-1]
    print(f"{i:<10} {str(np.round(our_sorted, 3)):>35} {str(np.round(sk_sorted, 3)):>35}")

### 7.3 Visual Comparison

In [ ]:
# Create sklearn GMM wrapper for plotting
class SklearnGMMWrapper:
    def __init__(self, gmm):
        self.gmm = gmm
        self.n_components = gmm.n_components
        self.weights_ = gmm.weights_
        self.means_ = gmm.means_
        self.covariances_ = gmm.covariances_
        self.covariance_type = gmm.covariance_type
    
    def predict(self, X):
        return self.gmm.predict(X)
    
    def predict_proba(self, X):
        return self.gmm.predict_proba(X)


fig, axes = plt.subplots(1, 2, figsize=(14, 6))

plot_gmm_results(X_elongated, our_gmm, axes[0], 'Our GMM Implementation')
plot_gmm_results(X_elongated, SklearnGMMWrapper(sklearn_gmm), axes[1], 'sklearn GaussianMixture')

plt.tight_layout()
plt.show()

### 7.4 Performance Comparison

In [ ]:
import time

# Generate larger dataset for timing
X_large, _ = make_blobs(n_samples=5000, centers=5, cluster_std=1.0, random_state=42)

# Time our implementation
start = time.time()
our_gmm_large = GaussianMixtureModel(
    n_components=5, covariance_type='full',
    max_iter=100, n_init=1, random_state=42
)
our_gmm_large.fit(X_large)
our_time = time.time() - start

# Time sklearn
start = time.time()
sklearn_gmm_large = GaussianMixture(
    n_components=5, covariance_type='full',
    max_iter=100, n_init=1, random_state=42
)
sklearn_gmm_large.fit(X_large)
sklearn_time = time.time() - start

print("="*60)
print("PERFORMANCE COMPARISON (5000 samples, 5 components)")
print("="*60)
print(f"\nOur implementation: {our_time:.4f} seconds")
print(f"sklearn:            {sklearn_time:.4f} seconds")
print(f"\nSpeed ratio (sklearn/ours): {our_time/sklearn_time:.2f}x")

print(f"\nNote: sklearn is optimized with Cython; our pure NumPy")
print(f"implementation is expected to be slower but more educational.")

### 7.5 Summary and Best Practices

In [ ]:
print("""
================================================================================
GAUSSIAN MIXTURE MODELS: SUMMARY AND BEST PRACTICES
================================================================================

KEY TAKEAWAYS:

1. GMM vs K-Means:
   - GMM provides soft clustering (probabilities) vs hard clustering
   - GMM handles elliptical clusters; K-Means assumes spherical
   - GMM is a generative model; can sample new data

2. Covariance Types:
   - 'full': Most flexible, but most parameters (O(D^2) per component)
   - 'diag': Good balance; axis-aligned ellipses (O(D) per component)
   - 'spherical': Equivalent to K-Means with soft assignments
   - 'tied': All components share same shape

3. Model Selection:
   - BIC: More conservative, penalizes complexity
   - AIC: Less conservative, may overfit
   - Cross-validation: Most reliable but computationally expensive

4. Initialization:
   - Use n_init > 1 (typically 3-10) for robustness
   - K-Means++ initialization often works well
   - Watch for degenerate solutions (empty clusters)

5. Convergence:
   - Monitor log-likelihood; should monotonically increase
   - Set reasonable tol (1e-3 to 1e-4) and max_iter (100-300)
   - Check converged_ attribute

6. Practical Tips:
   - Standardize features before fitting
   - Use regularization (reg_covar) for numerical stability
   - For high-D data, consider 'diag' covariance or PCA first
   - For production, use sklearn (optimized); use this for learning

================================================================================
""")

In [ ]:
# Final comprehensive example
print("Final Example: Complete GMM Pipeline")
print("="*60)

# 1. Generate data
X_final, y_final = generate_elongated_clusters(n_samples=500, random_state=123)

# 2. Standardize (important for GMM)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_final)

# 3. Model selection
selection = select_n_components(X_scaled, max_components=6)
best_k = selection['n_components'][np.argmin(selection['bic'])]
print(f"Best number of components (BIC): {best_k}")

# 4. Fit final model
final_gmm = GaussianMixtureModel(
    n_components=best_k,
    covariance_type='full',
    max_iter=100,
    n_init=5,
    random_state=42
)
final_gmm.fit(X_scaled)

print(f"Converged: {final_gmm.converged_}")
print(f"Iterations: {final_gmm.n_iter_}")
print(f"Final BIC: {final_gmm.bic(X_scaled):.2f}")

# 5. Evaluate
labels = final_gmm.predict(X_scaled)
accuracy = cluster_accuracy(y_final, labels)
print(f"Clustering accuracy: {accuracy:.2%}")

# 6. Visualize
fig, ax = plt.subplots(figsize=(10, 8))

# Transform means back for visualization
means_original = scaler.inverse_transform(final_gmm.means_)

# Plot in original scale
ax.scatter(X_final[:, 0], X_final[:, 1], c=labels, cmap='viridis',
          alpha=0.6, edgecolors='white', s=50)
ax.scatter(means_original[:, 0], means_original[:, 1], c='red',
          marker='X', s=300, edgecolors='black', linewidths=2,
          label='Cluster Centers')

ax.set_title(f'GMM Clustering (K={best_k}, Accuracy={accuracy:.2%})', fontsize=14)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend()

plt.tight_layout()
plt.show()

print("\nNotebook complete!")